# Hands-on Lab: Generating Synthetic QR Codes with a GAN

---

### Overview

This lab implements a Generative Adversarial Network (GAN) to produce synthetic QR code images. A GAN consists of two neural networks trained in opposition: a **Generator** that learns to produce realistic images from random noise, and a **Discriminator** that learns to distinguish real images from generated ones. As training progresses, the generator improves until its outputs become difficult to tell apart from real QR codes.

### Learning Objectives
1. Understand the GAN training loop — adversarial optimisation between generator and discriminator
2. Build and connect a Conv2DTranspose generator and Conv2D discriminator in Keras
3. Preprocess image data correctly for GAN training (`[-1, 1]` normalisation)
4. Recognise why GANs matter as a cybersecurity threat vector

## Step 1: Install and Import Required Libraries

In [ ]:
!pip install qrcode[pil] tensorflow numpy pillow matplotlib --quiet

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import qrcode

from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.layers import (
    Dense, Reshape, LeakyReLU, BatchNormalization,
    Conv2DTranspose, Conv2D, Flatten, Dropout
)
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.preprocessing.image import load_img, img_to_array

import warnings
warnings.filterwarnings('ignore')

## Step 2: Create Directory Structure

In [ ]:
os.makedirs('qr_codes/train',     exist_ok=True)
os.makedirs('qr_codes/val',       exist_ok=True)
os.makedirs('qr_codes/generated', exist_ok=True)
os.makedirs('gan_checkpoints',    exist_ok=True)

print("Directory structure created.")

## Step 3: Generate QR Code Dataset

The dataset is generated programmatically — 80 training QR codes and 20 validation QR codes, each encoding a unique message string.

In [ ]:
def generate_qr_code(data, filename):
    """Generate a single QR code image and save it to disk."""
    qr = qrcode.QRCode(version=1, box_size=10, border=5)
    qr.add_data(data)
    qr.make(fit=True)
    img = qr.make_image(fill_color='black', back_color='white')
    img.save(filename)

# 80 training QR codes
for i in range(80):
    generate_qr_code(f"Message {i}", f"qr_codes/train/qr_code_{i}.png")

# 20 validation QR codes
for i in range(20):
    generate_qr_code(f"Message {i + 80}", f"qr_codes/val/qr_code_{i}.png")

print("QR code generation complete.")

## Step 4: Verify QR Code Generation

In [ ]:
train_filenames = os.listdir('qr_codes/train')
val_filenames   = os.listdir('qr_codes/val')

print(f"Training images generated:   {len(train_filenames)}")
print(f"Validation images generated: {len(val_filenames)}")

## Step 5: Load and Preprocess Images

Images are normalised to `[-1, 1]` — this matches the `tanh` output activation of the generator. Using `[0, 1]` normalisation with a `tanh` generator is a common mismatch that causes training instability.

In [ ]:
def load_images(image_dir, target_size=(100, 100)):
    """Load all images from a directory, resize, and return as a numpy array."""
    images = []
    for filename in os.listdir(image_dir):
        img       = load_img(os.path.join(image_dir, filename), target_size=target_size)
        img_array = img_to_array(img)
        images.append(img_array)
    return np.array(images)

train_images = load_images('qr_codes/train')
val_images   = load_images('qr_codes/val')

# Normalise pixel values from [0, 255] to [-1, 1] to match generator tanh output
train_images = (train_images - 127.5) / 127.5
val_images   = (val_images   - 127.5) / 127.5

print(f"Training images shape:   {train_images.shape}")
print(f"Validation images shape: {val_images.shape}")

## Step 6: Build the Generator

The generator maps a 100-dimensional noise vector to a 100×100×3 image. `Conv2DTranspose` layers progressively upsample: 25×25 → 50×50 → 100×100. `tanh` output keeps pixel values in `[-1, 1]`.

In [ ]:
def build_generator():
    model = Sequential()

    model.add(Dense(256 * 25 * 25, input_dim=100))
    model.add(LeakyReLU(alpha=0.2))
    model.add(Reshape((25, 25, 256)))

    model.add(Conv2DTranspose(128, (4, 4), strides=(2, 2), padding='same'))
    model.add(LeakyReLU(alpha=0.2))
    model.add(BatchNormalization())

    model.add(Conv2DTranspose(64, (4, 4), strides=(2, 2), padding='same'))
    model.add(LeakyReLU(alpha=0.2))
    model.add(BatchNormalization())

    model.add(Conv2DTranspose(3, (4, 4), activation='tanh', padding='same'))

    return model

generator = build_generator()
generator.summary()

## Step 7: Build the Discriminator

The discriminator is a binary image classifier — real (1) vs fake (0). Three Conv2D layers with increasing filters progressively extract features, with Dropout preventing the discriminator from becoming too dominant too quickly.

In [ ]:
def build_discriminator():
    model = Sequential()

    model.add(Conv2D(64,  (3, 3), padding='same', input_shape=(100, 100, 3)))
    model.add(LeakyReLU(alpha=0.2))
    model.add(Dropout(0.4))

    model.add(Conv2D(128, (3, 3), strides=(2, 2), padding='same'))
    model.add(LeakyReLU(alpha=0.2))
    model.add(Dropout(0.4))

    model.add(Conv2D(256, (3, 3), strides=(2, 2), padding='same'))
    model.add(LeakyReLU(alpha=0.2))
    model.add(Dropout(0.4))

    model.add(Flatten())
    model.add(Dense(1, activation='sigmoid'))

    return model

discriminator = build_discriminator()
discriminator.compile(
    loss='binary_crossentropy',
    optimizer=Adam(0.0002, 0.5),
    metrics=['accuracy']
)
discriminator.summary()

## Step 8: Build the Combined GAN Model

The discriminator is frozen during GAN training — only the generator's weights are updated. This allows the generator to learn to fool the discriminator without the discriminator adapting simultaneously.

In [ ]:
def build_gan(generator, discriminator):
    discriminator.trainable = False
    gan_input  = generator.input
    gan_output = discriminator(generator.output)
    gan = Model(gan_input, gan_output)
    gan.compile(loss='binary_crossentropy', optimizer=Adam(0.0002, 0.5))
    return gan

gan = build_gan(generator, discriminator)
gan.summary()

## Step 9: Train the GAN

Each epoch alternates between:
1. **Discriminator update** — trained on real images (label=1) and generated fakes (label=0)
2. **Generator update** — trained through the frozen discriminator with target label=1 (fool it into thinking fakes are real)

Checkpoints are saved every 100 epochs so training can be resumed or rolled back.

In [ ]:
def train_gan(generator, discriminator, gan, train_images,
              epochs=5000, batch_size=32, checkpoint_interval=100):
    """
    Train the GAN.

    Parameters
    ----------
    train_images : np.ndarray
        Normalised training images in [-1, 1], shape (n, h, w, c).
        Passed explicitly to avoid silent dependence on a global variable.
    """
    checkpoint_dir = 'gan_checkpoints'

    for epoch in range(epochs):

        # --- Discriminator update ---
        idx         = np.random.randint(0, train_images.shape[0], batch_size)
        real_images = train_images[idx]

        noise       = np.random.normal(0, 1, (batch_size, 100))
        # verbose=0 suppresses per-batch progress bars inside the training loop
        fake_images = generator.predict(noise, verbose=0)

        d_loss_real = discriminator.train_on_batch(real_images, np.ones((batch_size, 1)))
        d_loss_fake = discriminator.train_on_batch(fake_images, np.zeros((batch_size, 1)))
        d_loss      = 0.5 * np.add(d_loss_real, d_loss_fake)

        # --- Generator update ---
        noise  = np.random.normal(0, 1, (batch_size, 100))
        g_loss = gan.train_on_batch(noise, np.ones((batch_size, 1)))

        # Progress log every 100 epochs
        if epoch % 100 == 0:
            print(f"Epoch {epoch:>4d} | "
                  f"D loss: {d_loss[0]:.4f} | D acc: {d_loss[1]:.4f} | "
                  f"G loss: {g_loss:.4f}")

        # Save checkpoints
        if epoch % checkpoint_interval == 0:
            generator.save(
                os.path.join(checkpoint_dir, f"generator_epoch_{epoch}.h5"))
            discriminator.save(
                os.path.join(checkpoint_dir, f"discriminator_epoch_{epoch}.h5"))


train_gan(generator, discriminator, gan, train_images, epochs=5000, batch_size=32)

## Step 10: Generate and Display Synthetic QR Codes

In [ ]:
def generate_and_save_qr_codes(generator, n_samples=10):
    """Generate synthetic QR code images from random noise and save to disk."""
    noise            = np.random.normal(0, 1, (n_samples, 100))
    generated_images = generator.predict(noise, verbose=0)

    # Rescale from [-1, 1] back to [0, 1] for display
    generated_images = 0.5 * generated_images + 0.5

    for i in range(n_samples):
        plt.imshow(generated_images[i])
        plt.axis('off')
        plt.savefig(f"qr_codes/generated/qr_code_{i}.png", bbox_inches='tight')
        plt.close()

    print(f"{n_samples} synthetic QR codes saved to qr_codes/generated/")

generate_and_save_qr_codes(generator, n_samples=10)

In [ ]:
# Side-by-side comparison: one real vs one generated QR code
fig, axes = plt.subplots(1, 2, figsize=(8, 4))

real_img = load_img('qr_codes/train/qr_code_0.png', target_size=(100, 100))
axes[0].imshow(img_to_array(real_img).astype('uint8'))
axes[0].set_title('Real QR Code')
axes[0].axis('off')

gen_img = load_img('qr_codes/generated/qr_code_0.png', target_size=(100, 100))
axes[1].imshow(img_to_array(gen_img).astype('uint8'))
axes[1].set_title('GAN-Generated QR Code')
axes[1].axis('off')

plt.suptitle('Real vs Generated — 5000 epochs, 80 training images', fontsize=11)
plt.tight_layout()
plt.show()

---

## Personal Analysis

### Why the Generated QR Codes Won't Scan

The GAN is trained on only 80 images for 5000 epochs — a regime almost certain to produce **mode collapse**, where the generator finds a small set of outputs that fool the discriminator and stops diversifying. Even without collapse, a QR code is a highly structured binary signal with Reed-Solomon error correction, finder patterns, alignment markers, and format information baked into precise pixel positions. A convolutional GAN learns statistical texture and colour distributions — it has no mechanism to enforce the rigid geometric constraints that make a QR code decodable.

### The Cybersecurity Relevance: GANs as an Adversarial Threat

- **Adversarial examples** — GANs can generate inputs that are visually indistinguishable from benign files but cause ML-based classifiers to misclassify them. A malware sample that looks like a JPEG to a neural network classifier is a real threat.
- **Deepfakes and social engineering** — synthetic faces, voices, and documents used in phishing and identity fraud campaigns are GAN outputs.
- **Evading malware detection** — the same opcode-sequence modelling from the HMM lab combined with a GAN-style generator could theoretically produce metamorphic variants that fool ML-based antivirus.
- **Data augmentation for defenders** — conversely, defenders use GANs to generate synthetic malware samples to augment limited training datasets, improving detection model robustness.

### The Training Loop Design

One implementation decision worth highlighting: the discriminator uses `Adam(0.0002, 0.5)` in the combined GAN but is compiled separately with `optimizer='adam'` (default learning rate 0.001) in the standalone discriminator. This asymmetry is intentional — the discriminator trains faster with a higher learning rate when evaluated alone, but needs a slower, matched rate during joint training to avoid outpacing the generator. A discriminator that becomes too strong too early causes **vanishing gradients** in the generator, which then receives no useful signal for improvement.